# DS5 Step 2 — Path C: Network-Structural Sub-Phenotyping

**Stage:** Stage 4, Step 2 — Behavioral Subgroup Discovery (Path C of six)
**Date:** 2026-04-19
**Blueprint:** `outputs/reports/stage4/ds5_path_c_network_subphenotype_blueprint.md`
**Depends on:** `100_phenotype_exploration.ipynb` (Phase 1 cluster assignments)

---

## Goal

Phase 1 clustering (notebook 100) found a stable k=2 solution where the split was almost entirely
driven by NeT features (HHI_net, rho_self). Profile B (N=289, 70%) occupies the near-ceiling region
of both features. Profile A (N=125, 30%) has genuine network diversity.

**This notebook asks:** Is Profile B a single homogeneous behavioral type, or does it contain
meaningful sub-types when examined exclusively with NeT features?

## What this notebook does NOT do
- No intervention prescription (Step 3)
- No condition used as clustering feature
- No causal claims from temporal succession networks
- No claims about validated traits — all findings are exploratory, DS5-specific

## Key caveats throughout this notebook
- DS5 networks are built from OneSec-triggered events only, not total app usage
- Temporal succession (app_i → app_j) is not causal coupling
- First-order Markov property is not verified for DS5
- Stationary distribution π is an approximation given short windows (7–45 days)
- All network claims are descriptive structural observations, not behavioral laws

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score
)
from scipy.stats import kruskal, chi2_contingency
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# --- Paths ---
ROOT = Path('..') 
PHENOTYPING_DIR = ROOT / 'outputs' / 'model_outputs' / 'phenotyping'
FIGURES_DIR = PHENOTYPING_DIR / 'figures'
NET_DIR = ROOT / 'outputs' / 'model_outputs' / 'network_models'
STOP_DIR = ROOT / 'outputs' / 'model_outputs' / 'stochastic_processes'
INT_DIR = ROOT / 'outputs' / 'model_outputs' / 'information_theory'
STM_DIR = ROOT / 'outputs' / 'model_outputs' / 'statistical_mechanics'
FEATURES_DIR = ROOT / 'outputs' / 'features' / 'ds5'

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Bootstrap config
N_BOOTSTRAPS = 100
SAMPLE_FRAC = 0.8
MIN_CLUSTER_SIZE = 30
BONFERRONI_N = 5  # 5 psychosocial scales
ALPHA_BONFERRONI = 0.05 / BONFERRONI_N

print(f'Output directory: {PHENOTYPING_DIR.resolve()}')
print(f'Bonferroni threshold (5 tests): α = {ALPHA_BONFERRONI:.4f}')

## 1. Load and merge data

In [ ]:
# Load Phase 1 cluster assignments
phase1 = pd.read_csv(PHENOTYPING_DIR / 'ds5_cluster_assignments.csv')
print(f'Phase 1 assignments: {len(phase1)} rows')
print('Profile counts:', phase1['profile_p1'].value_counts().to_dict())

# Load NeT features
net = pd.read_csv(NET_DIR / 'participant_network_metrics.csv')
print(f'\nNeT metrics: {len(net)} rows')
print('NeT-eligible:', net['is_net_eligible'].sum())
print('Stationary-eligible:', net['is_stationary_eligible'].sum())

# Load StoP features (overlay)
stop = pd.read_csv(STOP_DIR / 'participant_parameters.csv')
print(f'\nStoP parameters: {len(stop)} rows')

# Load InT features (H_emp overlay)
int_df = pd.read_csv(INT_DIR / 'participant_entropy_metrics.csv')
print(f'InT entropy metrics: {len(int_df)} rows')

# Load StM features (chi_behav overlay)
stm = pd.read_csv(STM_DIR / 'participant_stm_parameters.csv')
print(f'StM parameters: {len(stm)} rows')

# Load psychosocial scales
survey = pd.read_csv(FEATURES_DIR / 'ds5_survey_features.csv')
PSYCH_SCALES = ['fomo_mean_pre', 'belong_mean_pre', 'ostracism_mean_pre',
                'digital_wellbeing_pre', 'social_media_wellbeing_pre']
print(f'Survey features: {len(survey)} rows')

In [ ]:
# Build master dataframe: Phase 1 labels + NeT + overlays
# Start from phase1 (N=414, complete-case for Phase 1)
df = phase1.copy()

# Merge NeT features
net_cols = ['participant_id_canonical', 'HHI_net', 'top1_share_net', 'top3_share_net',
            'rho_self', 'H_trans', 'H_pi', 'I_consec', 'H_app',
            'is_net_eligible', 'is_stationary_eligible']
df = df.merge(net[net_cols], on='participant_id_canonical', how='left')

# Merge StoP overlay features
stop_cols = ['participant_id_canonical', 'sig_hat', 'mu_hat', 'x_mean', 'sigma_mu_ratio']
df = df.merge(stop[stop_cols], on='participant_id_canonical', how='left')

# Merge InT overlay (H_emp)
int_cols = ['participant_id_canonical', 'H_emp', 'H_out', 'H_hour']
df = df.merge(int_df[int_cols], on='participant_id_canonical', how='left')

# Merge StM overlay (chi_behav)
stm_cols = ['participant_id_canonical', 'chi_behav', 'four_model_convergent', 'high_chi']
df = df.merge(stm[stm_cols], on='participant_id_canonical', how='left')

# Merge psychosocial
df = df.merge(survey[['participant_id_canonical'] + PSYCH_SCALES], on='participant_id_canonical', how='left')

print(f'Master dataframe: {df.shape}')
print('Profile counts:', df['profile_p1'].value_counts().to_dict())
print('NeT-eligible in master:', df['is_net_eligible'].sum())

In [ ]:
# Define analysis populations
# Primary: Profile B, NeT-eligible, complete on [HHI_net, rho_self, H_trans, H_app]
NET_FEATURES = ['HHI_net', 'rho_self', 'H_trans', 'H_app']

mask_B = (df['profile_p1'] == 'Profile B') & (df['is_net_eligible'] == True)
profB = df[mask_B].copy().dropna(subset=NET_FEATURES).reset_index(drop=True)

mask_A = (df['profile_p1'] == 'Profile A') & (df['is_net_eligible'] == True)
profA = df[mask_A].copy().dropna(subset=NET_FEATURES).reset_index(drop=True)

# Full NeT-eligible sample (not split by Phase 1)
full_net = net[net['is_net_eligible'] == True].copy().dropna(subset=NET_FEATURES).reset_index(drop=True)
# Add Phase 1 labels to full_net for ARI comparison
full_net = full_net.merge(phase1[['participant_id_canonical', 'profile_p1']],
                          on='participant_id_canonical', how='left')

print(f'Profile B analysis population (NeT-eligible, complete): N = {len(profB)}')
print(f'Profile A analysis population (NeT-eligible, complete): N = {len(profA)}')
print(f'Full NeT-eligible sample (complete on NeT features): N = {len(full_net)}')
print(f'\nProfile B condition distribution:')
print(profB['study_condition'].value_counts())

---
## Section 1: Profile B — Variance Inspection and Decision Gate

Before sub-clustering, inspect the variance of NeT features within Profile B.

**Decision gate:** If all NeT features have std < 0.05 (near-homogeneous), clustering will be
noise-fitting rather than structure discovery — report homogeneity as a finding and stop.

**Important structural finding (identified during pre-analysis):**
H_trans and H_app are zero-inflated within Profile B. Many participants have exactly 0 for both,
meaning they have only one observed app in their OneSec event log (pure single-app users).
This creates a discrete, not continuous, structure. The sub-clustering is expected to recover
this discrete split.

In [ ]:
# --- Profile B: Feature-level variance inspection ---
print('=== Profile B: NeT Feature Variance Inspection ===')
print()
decision_gate_triggered = False
for col in NET_FEATURES + ['top1_share_net', 'H_pi', 'I_consec']:
    if col not in profB.columns:
        continue
    s = profB[col].dropna()
    at_ceil = (s >= 0.99).mean() if s.max() <= 1.0 else 0.0
    at_zero = (s == 0.0).mean()
    print(f'  {col:20s}: mean={s.mean():.4f}  std={s.std():.4f}  '  
          f'IQR=[{s.quantile(0.25):.3f}, {s.quantile(0.75):.3f}]  '
          f'pct_ceiling={at_ceil:.1%}  pct_zero={at_zero:.1%}')

print()
# Check decision gate: ALL primary features must have std < 0.05 to trigger
low_var_features = [c for c in NET_FEATURES if profB[c].std() < 0.05]
print(f'Features with std < 0.05: {low_var_features}')
if len(low_var_features) == len(NET_FEATURES):
    decision_gate_triggered = True
    print('DECISION GATE TRIGGERED: Profile B is near-homogeneous on ALL NeT features.')
    print('Sub-clustering would fit noise. Reporting homogeneity as the finding.')
else:
    print('DECISION GATE NOT TRIGGERED: Sufficient variance exists for sub-clustering.')
    print(f'  Features with real variance (std >= 0.05): '
          f'{[c for c in NET_FEATURES if profB[c].std() >= 0.05]}')

# Zero-inflation assessment
print()
pct_htrans_zero = (profB['H_trans'] == 0).mean()
pct_happ_zero = (profB['H_app'] == 0).mean()
n_zero = (profB['H_trans'] == 0).sum()
n_nonzero = (profB['H_trans'] > 0).sum()
print(f'Zero-inflation assessment:')
print(f'  H_trans == 0: {n_zero} participants ({pct_htrans_zero:.1%})')
print(f'  H_app == 0:   {pct_happ_zero:.1%} (same participants)')
print(f'  H_trans > 0:  {n_nonzero} participants ({1-pct_htrans_zero:.1%})')
print()
print('  Interpretation: Participants with H_trans=H_app=0 observed only one app')
print('  across all OneSec-triggered events. This is a discrete structural distinction,')
print('  not a continuous cluster boundary.')

In [ ]:
# --- Figure C1: Profile B NeT feature distributions ---
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
fig.suptitle('Profile B — NeT Feature Distributions\n'
             '(hub-dominated participants, N=289)', fontsize=13, fontweight='bold')

for ax, col, label in zip(
    axes.ravel(),
    ['HHI_net', 'rho_self', 'H_trans', 'H_app'],
    ['HHI_net (network flow concentration)',
     'rho_self (self-loop / same-app return rate)',
     'H_trans (conditional transition entropy)',
     'H_app (app diversity entropy)']
):
    vals = profB[col].dropna()
    ax.hist(vals, bins=40, color='#4472C4', alpha=0.75, edgecolor='white', linewidth=0.3)
    ax.axvline(vals.median(), color='#C00000', linestyle='--', linewidth=1.5,
               label=f'Median={vals.median():.3f}')
    ax.set_xlabel(col, fontsize=10)
    ax.set_ylabel('Count', fontsize=9)
    ax.set_title(label, fontsize=9)
    ax.legend(fontsize=8)
    # Annotate zero mass for H_trans and H_app
    if col in ['H_trans', 'H_app']:
        pct_z = (vals == 0).mean()
        ax.text(0.98, 0.92, f'{pct_z:.0%} at 0', transform=ax.transAxes,
                ha='right', fontsize=9, color='darkred',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_c1_profileB_net_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_c1_profileB_net_distributions.png')

In [ ]:
# --- Profile B: PCA to assess effective dimensionality ---
scaler_B = StandardScaler()
X_B_scaled = scaler_B.fit_transform(profB[NET_FEATURES])

pca_B = PCA(n_components=4)
X_B_pca = pca_B.fit_transform(X_B_scaled)

print('=== Profile B: PCA — Effective Dimensionality ===')
ev_ratio = pca_B.explained_variance_ratio_
for i, (ev, cum) in enumerate(zip(ev_ratio, np.cumsum(ev_ratio))):
    print(f'  PC{i+1}: {ev:.3f} ({cum:.3f} cumulative)')

n_comp_80 = np.argmax(np.cumsum(ev_ratio) >= 0.80) + 1
print(f'  Components needed for 80% variance: {n_comp_80}')

print()
print('PCA loadings (contribution of each NeT feature per component):')
loadings_B = pd.DataFrame(
    pca_B.components_.T,
    index=NET_FEATURES,
    columns=[f'PC{i+1}' for i in range(4)]
)
print(loadings_B.round(3))

In [ ]:
# --- Figure C2: Profile B PCA biplot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Profile B — PCA Analysis\n'
             '(NeT features: HHI_net, rho_self, H_trans, H_app)', fontsize=12)

# Left: scree plot
ax = axes[0]
components = np.arange(1, 5)
ax.bar(components, ev_ratio * 100, color='#4472C4', alpha=0.8)
ax.plot(components, np.cumsum(ev_ratio) * 100, 'o-', color='#C00000', linewidth=2)
ax.axhline(80, color='gray', linestyle='--', linewidth=1, label='80% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('Scree Plot')
ax.legend(fontsize=9)
ax.set_xticks(components)
for i, (v, cv) in enumerate(zip(ev_ratio * 100, np.cumsum(ev_ratio) * 100)):
    ax.text(i + 1, v + 1, f'{v:.0f}%', ha='center', fontsize=9)

# Right: PC1 vs PC2 scatter, colored by H_trans value
ax = axes[1]
sc = ax.scatter(X_B_pca[:, 0], X_B_pca[:, 1],
                c=profB['H_trans'], cmap='viridis', alpha=0.6, s=20)
plt.colorbar(sc, ax=ax, label='H_trans value')

# Loading arrows
scale = 3.0
for feat, (l1, l2) in zip(NET_FEATURES, pca_B.components_.T[:, :2]):
    ax.annotate('', xy=(l1 * scale, l2 * scale), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
    ax.text(l1 * scale * 1.1, l2 * scale * 1.1, feat, fontsize=8, color='darkred')

ax.set_xlabel(f'PC1 ({ev_ratio[0]*100:.0f}% var)')
ax.set_ylabel(f'PC2 ({ev_ratio[1]*100:.0f}% var)')
ax.set_title('PC1 vs PC2 (colored by H_trans)')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_c2_profileB_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_c2_profileB_pca.png')

---
## Section 2: Profile B — Sub-cluster Selection

**Feature note:** HHI_net and rho_self are near-ceiling for most Profile B participants.
H_trans and H_app are zero-inflated (70.6% at exactly 0), creating a discrete sub-structure.
StandardScaler will amplify variance equally — the H_trans/H_app axis will dominate
clustering because it has the real discriminating signal.

**All four features are used** to remain consistent with the theoretically motivated feature space.
The PCA (above) will confirm which features are driving the components.

In [ ]:
# Bootstrap Jaccard stability function (same as Phase 1)
def bootstrap_jaccard_stability(X, k, n_bootstraps=100, sample_frac=0.8, random_state=42):
    """Compute Jaccard stability of k-means clustering via bootstrapping.
    
    For each bootstrap, sample sample_frac of observations, re-cluster,
    and compute Jaccard coefficient of pairwise co-memberships vs. full solution.
    """
    rng = np.random.RandomState(random_state)
    n = len(X)
    sample_n = int(n * sample_frac)

    km_full = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    labels_full = km_full.fit_predict(X)

    jaccard_scores = []
    for _ in range(n_bootstraps):
        idx = rng.choice(n, size=sample_n, replace=False)
        X_boot = X[idx]
        km_boot = KMeans(n_clusters=k, init='k-means++', n_init=10,
                         random_state=int(rng.randint(10000)))
        labels_boot = km_boot.fit_predict(X_boot)
        L_full = labels_full[idx]
        L_boot = labels_boot
        tril_i, tril_j = np.tril_indices(sample_n, k=-1)
        sf = L_full[tril_i] == L_full[tril_j]
        sb = L_boot[tril_i] == L_boot[tril_j]
        tp = (sf & sb).sum()
        fp = (~sf & sb).sum()
        fn = (sf & ~sb).sum()
        denom = tp + fp + fn
        jaccard_scores.append(tp / denom if denom > 0 else 0.0)

    return float(np.mean(jaccard_scores)), float(np.std(jaccard_scores))

In [ ]:
# --- Profile B: k-selection loop ---
print('=== Profile B: Sub-cluster Selection (k = 2, 3, 4) ===')
print(f'Analysis population: N = {len(profB)}')
print()

X_B = X_B_scaled  # Already scaled above

B_metrics = []
for k in [2, 3, 4]:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_B)
    sizes = np.bincount(labels)
    min_size = sizes.min()
    sil = silhouette_score(X_B, labels)
    ch = calinski_harabasz_score(X_B, labels)
    db = davies_bouldin_score(X_B, labels)
    B_metrics.append({
        'k': k, 'silhouette': sil, 'CH': ch, 'DB': db,
        'min_size': min_size, 'sizes': sizes.tolist(),
        'size_ok': min_size >= MIN_CLUSTER_SIZE
    })
    flag = '' if min_size >= MIN_CLUSTER_SIZE else f'  ⚠ min cluster size < {MIN_CLUSTER_SIZE}'
    print(f'  k={k}: silhouette={sil:.3f}  CH={ch:.1f}  DB={db:.3f}  '
          f'min_size={min_size}  sizes={sizes.tolist()}{flag}')

B_metrics_df = pd.DataFrame(B_metrics)

# Select best k: highest silhouette among solutions with min_size >= 30
valid = B_metrics_df[B_metrics_df['size_ok'] == True]
if len(valid) == 0:
    print('\nNo valid k with min_size >= 30. Using k=2.')
    best_k_B = 2
else:
    best_k_B = int(valid.loc[valid['silhouette'].idxmax(), 'k'])
    
print(f'\nSelected k for Profile B: k = {best_k_B}')

In [ ]:
# --- Bootstrap Jaccard stability for Profile B best k ---
print(f'=== Profile B: Bootstrap Jaccard Stability (k={best_k_B}, {N_BOOTSTRAPS} bootstraps) ===')
jacc_mean_B, jacc_std_B = bootstrap_jaccard_stability(
    X_B, best_k_B, n_bootstraps=N_BOOTSTRAPS,
    sample_frac=SAMPLE_FRAC, random_state=RANDOM_STATE
)
stability_label_B = 'STABLE' if jacc_mean_B >= 0.75 else ('MODERATE' if jacc_mean_B >= 0.5 else 'UNSTABLE')
print(f'  Jaccard mean = {jacc_mean_B:.3f}  std = {jacc_std_B:.3f}  → {stability_label_B}')

# Also compute for k=2 if best_k_B != 2
stability_records = [{
    'population': 'Profile B',
    'k': best_k_B,
    'jaccard_mean': jacc_mean_B,
    'jaccard_std': jacc_std_B,
    'stability': stability_label_B
}]
if best_k_B != 2:
    j2_mean, j2_std = bootstrap_jaccard_stability(X_B, 2, n_bootstraps=N_BOOTSTRAPS,
                                                   sample_frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
    s2_label = 'STABLE' if j2_mean >= 0.75 else 'MODERATE'
    print(f'  Also k=2: Jaccard mean = {j2_mean:.3f}  std = {j2_std:.3f}  → {s2_label}')
    stability_records.append({
        'population': 'Profile B', 'k': 2,
        'jaccard_mean': j2_mean, 'jaccard_std': j2_std, 'stability': s2_label
    })

In [ ]:
# --- Figure C3: Profile B cluster selection metrics ---
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle(f'Profile B — Sub-cluster Selection Metrics\n(N={len(profB)})', fontsize=12)

ks = B_metrics_df['k']
colors_k = ['#4472C4' if k == best_k_B else '#A9C4E8' for k in ks]

axes[0].bar(ks, B_metrics_df['silhouette'], color=colors_k)
axes[0].set_xlabel('k'); axes[0].set_ylabel('Silhouette Score')
axes[0].set_title('Silhouette (↑ better)')
axes[0].set_xticks(ks)
for i, (k, v) in enumerate(zip(ks, B_metrics_df['silhouette'])):
    axes[0].text(k, v + 0.002, f'{v:.3f}', ha='center', fontsize=9)

axes[1].bar(ks, B_metrics_df['CH'], color=colors_k)
axes[1].set_xlabel('k'); axes[1].set_ylabel('Calinski-Harabasz')
axes[1].set_title('CH Score (↑ better)')
axes[1].set_xticks(ks)

axes[2].bar(ks, B_metrics_df['DB'], color=colors_k)
axes[2].set_xlabel('k'); axes[2].set_ylabel('Davies-Bouldin')
axes[2].set_title('DB Score (↓ better)')
axes[2].set_xticks(ks)

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Annotate selected k
best_row = B_metrics_df[B_metrics_df['k'] == best_k_B].iloc[0]
fig.text(0.5, -0.02,
         f'Selected: k={best_k_B}  (silhouette={best_row["silhouette"]:.3f}, '
         f'Jaccard={jacc_mean_B:.3f} ± {jacc_std_B:.3f} — {stability_label_B})',
         ha='center', fontsize=10, style='italic')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_c3_profileB_subcluster_selection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_c3_profileB_subcluster_selection.png')

---
## Section 3: Profile B — Sub-cluster Profiles and Overlays

In [ ]:
# --- Fit final Profile B sub-clustering ---
km_B_final = KMeans(n_clusters=best_k_B, init='k-means++', n_init=10, random_state=RANDOM_STATE)
profB = profB.copy()
profB['subcluster_B'] = km_B_final.fit_predict(X_B)

# Assign interpretive sub-cluster names based on H_trans median per cluster
cluster_htrans = profB.groupby('subcluster_B')['H_trans'].median().sort_values()
label_map = {}
if best_k_B == 2:
    ordered = cluster_htrans.index.tolist()
    label_map[ordered[0]] = 'B1'
    label_map[ordered[1]] = 'B2'
elif best_k_B == 3:
    ordered = cluster_htrans.index.tolist()
    label_map[ordered[0]] = 'B1'
    label_map[ordered[1]] = 'B2'
    label_map[ordered[2]] = 'B3'
else:
    for i, idx in enumerate(cluster_htrans.index):
        label_map[idx] = f'B{i+1}'
profB['subprofile_B'] = profB['subcluster_B'].map(label_map)

print(f'Profile B sub-cluster sizes:')
for sp, count in profB['subprofile_B'].value_counts().sort_index().items():
    pct = count / len(profB)
    print(f'  {sp}: N={count} ({pct:.1%})')

In [ ]:
# --- Profile B: Full feature profiles ---
ALL_PROFILE_COLS = NET_FEATURES + ['H_pi', 'I_consec', 'top1_share_net',
                                    'sig_hat', 'H_emp', 'x_mean', 'H_out',
                                    'chi_behav', 'H_hour', 'mu_hat']
ALL_PROFILE_COLS = [c for c in ALL_PROFILE_COLS if c in profB.columns]

profile_rows = []
for sp in sorted(profB['subprofile_B'].unique()):
    grp = profB[profB['subprofile_B'] == sp]
    row = {'subprofile': sp, 'N': len(grp)}
    for col in ALL_PROFILE_COLS:
        vals = grp[col].dropna()
        row[f'{col}_mean'] = vals.mean()
        row[f'{col}_median'] = vals.median()
        row[f'{col}_std'] = vals.std()
    profile_rows.append(row)

B_profiles_df = pd.DataFrame(profile_rows)

# Print readable summary
print('=== Profile B: Sub-cluster Feature Profiles (NeT features + overlays) ===')
display_cols = ['subprofile', 'N'] + [f'{c}_mean' for c in NET_FEATURES] + \
               [f'{c}_median' for c in ['H_pi', 'I_consec', 'sig_hat', 'H_emp', 'x_mean']]
display_cols = [c for c in display_cols if c in B_profiles_df.columns]
print(B_profiles_df[display_cols].to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# --- Figure C4: Profile B sub-cluster profiles (bar chart) ---
plot_features = NET_FEATURES + ['I_consec']
n_feat = len(plot_features)
subprofiles = sorted(profB['subprofile_B'].unique())
n_sp = len(subprofiles)

x = np.arange(n_feat)
width = 0.8 / n_sp
palette = ['#2E75B6', '#ED7D31', '#70AD47', '#FFC000'][:n_sp]

fig, ax = plt.subplots(figsize=(11, 5))
for i, sp in enumerate(subprofiles):
    row = B_profiles_df[B_profiles_df['subprofile'] == sp].iloc[0]
    n_sp_count = int(row['N'])
    means = [row[f'{f}_mean'] for f in plot_features]
    stds = [row[f'{f}_std'] for f in plot_features]
    bars = ax.bar(x + i * width - (n_sp - 1) * width / 2, means,
                  width=width, label=f'{sp} (N={n_sp_count})',
                  color=palette[i], alpha=0.85, edgecolor='white')
    ax.errorbar(x + i * width - (n_sp - 1) * width / 2, means,
                yerr=stds, fmt='none', color='black', capsize=3, linewidth=1.2)

ax.set_xticks(x)
ax.set_xticklabels(plot_features, fontsize=10)
ax.set_ylabel('Mean value', fontsize=11)
ax.set_title('Profile B — Network Sub-cluster Profiles (Mean ± SD)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(0, color='gray', linewidth=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add caution note
ax.text(0.99, 0.97,
        '⚠ DS5 networks from OneSec events only\nDescriptive; no causal coupling implied',
        transform=ax.transAxes, ha='right', va='top', fontsize=8,
        style='italic', color='gray')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_c4_profileB_subcluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_c4_profileB_subcluster_profiles.png')

In [ ]:
# --- Profile B: Condition distribution χ² test ---
print('=== Profile B: Condition Distribution Across Sub-clusters ===')
cond_tab = pd.crosstab(profB['subprofile_B'], profB['study_condition'])
print(cond_tab)
chi2, p_cond, dof, _ = chi2_contingency(cond_tab)
print(f'\nχ² = {chi2:.3f}, df = {dof}, p = {p_cond:.4f}')
if p_cond < 0.05:
    print('  ⚠ Condition distribution differs across sub-clusters (p < 0.05)')
    print('  ⚠ Sub-cluster profiles may partially reflect condition effects, not only behavioral traits')
else:
    print('  ✓ Condition distribution does not differ across sub-clusters (p ≥ 0.05)')
    print('  ✓ Sub-phenotypes cut across experimental conditions')

In [ ]:
# --- Profile B: Psychosocial Kruskal-Wallis tests ---
print('=== Profile B: Psychosocial Differentiation (Bonferroni-adjusted p vs α = 0.05, 5 tests) ===')
psych_results_B = []
groups_B = [profB[profB['subprofile_B'] == sp][s].dropna().values
            for sp in sorted(profB['subprofile_B'].unique())
            for s in []]

for scale in PSYCH_SCALES:
    if scale not in profB.columns:
        continue
    grp_vals = [profB[profB['subprofile_B'] == sp][scale].dropna().values
                for sp in sorted(profB['subprofile_B'].unique())]
    grp_vals = [g for g in grp_vals if len(g) >= 5]
    if len(grp_vals) < 2:
        continue
    H, p_raw = kruskal(*grp_vals)
    p_adj = min(p_raw * BONFERRONI_N, 1.0)
    sig = p_adj < 0.05  # p_adj is ALREADY Bonferroni-inflated (p_raw * 5); comparing it to 0.05/5 would apply the correction twice (matches notebook 102 cell 20)
    medians = {sp: profB[profB['subprofile_B'] == sp][scale].median()
               for sp in sorted(profB['subprofile_B'].unique())}
    print(f'  {scale:35s}: H={H:.3f}  p_raw={p_raw:.4f}  p_adj={p_adj:.4f}  '
          f'{"SIGNIFICANT" if sig else "ns"}')
    psych_results_B.append({
        'scale': scale, 'H_stat': H, 'p_raw': p_raw,
        'p_bonferroni': p_adj, 'significant': sig, 'medians': medians
    })

n_sig_B = sum(r['significant'] for r in psych_results_B)
print(f'\nSignificant psychosocial differences: {n_sig_B} of {len(psych_results_B)}')
if n_sig_B == 0:
    print('  No psychosocial differentiation across Profile B sub-clusters (after Bonferroni).')
    print('  The network sub-structure does not map onto pre-intervention psychological state.')

In [ ]:
# --- Profile B: Behavioral feature overlay (sig_hat, H_emp, x_mean) ---
print('=== Profile B: Behavioral Feature Overlay (NOT clustering features — descriptive only) ===')
overlay_features = ['sig_hat', 'H_emp', 'x_mean']

for feat in overlay_features:
    if feat not in profB.columns:
        continue
    sp_vals = [profB[profB['subprofile_B'] == sp][feat].dropna().values
               for sp in sorted(profB['subprofile_B'].unique())]
    sp_medians = {sp: profB[profB['subprofile_B'] == sp][feat].median()
                  for sp in sorted(profB['subprofile_B'].unique())}
    if len(sp_vals) >= 2 and all(len(v) >= 3 for v in sp_vals):
        H, p = kruskal(*sp_vals)
        sig_str = '(p<0.05)' if p < 0.05 else '(ns)'
    else:
        H, p, sig_str = np.nan, np.nan, ''
    median_str = '  '.join([f'{sp}={v:.3f}' for sp, v in sp_medians.items()])
    print(f'  {feat:10s}: {median_str}   KW H={H:.3f} p={p:.4f} {sig_str}')

print()
print('Note: Behavioral features were NOT used to define sub-clusters.')
print('Differences (if any) reflect co-variation between network structure and behavioral variability.')

In [ ]:
# --- Figure C5: Profile B behavioral overlay plots ---
overlay_features = ['sig_hat', 'H_emp', 'x_mean']
overlay_labels = {
    'sig_hat': 'σ̂ (behavioral diffusion / noise)',
    'H_emp': 'H_emp (behavioral entropy)',
    'x_mean': 'x̄ (mean override rate)'
}
subprofiles_sorted = sorted(profB['subprofile_B'].unique())
palette_sp = dict(zip(subprofiles_sorted, ['#2E75B6', '#ED7D31', '#70AD47', '#FFC000']))

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
fig.suptitle('Profile B — Behavioral Feature Overlay by Network Sub-cluster\n'
             '(These features were NOT used to define sub-clusters)', fontsize=11)

for ax, feat in zip(axes, overlay_features):
    data_groups = [profB[profB['subprofile_B'] == sp][feat].dropna()
                   for sp in subprofiles_sorted]
    bp = ax.violinplot(data_groups, positions=range(len(subprofiles_sorted)),
                       showmedians=True, showextrema=False)
    for pc, sp in zip(bp['bodies'], subprofiles_sorted):
        pc.set_facecolor(palette_sp[sp])
        pc.set_alpha(0.7)
    bp['cmedians'].set_color('black')
    ax.set_xticks(range(len(subprofiles_sorted)))
    ax.set_xticklabels([f'{sp}\n(N={len(profB[profB["subprofile_B"]==sp])})'
                        for sp in subprofiles_sorted], fontsize=9)
    ax.set_title(overlay_labels.get(feat, feat), fontsize=9)
    ax.set_ylabel(feat, fontsize=9)
    # KW p-value
    vals = [profB[profB['subprofile_B'] == sp][feat].dropna().values for sp in subprofiles_sorted]
    if all(len(v) >= 3 for v in vals):
        H, p = kruskal(*vals)
        ax.text(0.98, 0.97, f'KW p={p:.4f}', transform=ax.transAxes,
                ha='right', va='top', fontsize=8, style='italic',
                color='red' if p < 0.05 else 'gray')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_c5_profileB_overlay.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_c5_profileB_overlay.png')

---
## Section 4: Profile A — Secondary Sub-cluster Analysis

Profile A (N=125) has genuinely distributed NeT structure: all participants have H_trans > 0
and H_app > 0. The question is whether degrees of network diversity create further sub-types.

**Small N caution:** Profile A sub-clustering uses N≈115. Bootstrap stability is mandatory.
If stability is below 0.75, report as "no stable sub-structure found" rather than forcing clusters.

In [ ]:
# --- Profile A: Variance inspection ---
print('=== Profile A: NeT Feature Variance Inspection ===')
print(f'Analysis population: N = {len(profA)}')
print()
for col in NET_FEATURES:
    s = profA[col].dropna()
    print(f'  {col:20s}: mean={s.mean():.4f}  std={s.std():.4f}  '
          f'IQR=[{s.quantile(0.25):.3f}, {s.quantile(0.75):.3f}]')

# Check for zero inflation in Profile A
print(f'\nH_trans == 0 in Profile A: {(profA["H_trans"]==0).sum()} ({(profA["H_trans"]==0).mean():.1%})')
print('(Expected: 0 — Profile A has genuinely distributed networks)')

In [ ]:
# --- Profile A: Scale and k-selection ---
scaler_A = StandardScaler()
X_A = scaler_A.fit_transform(profA[NET_FEATURES])

print('=== Profile A: Sub-cluster Selection (k = 2, 3) ===')
A_metrics = []
for k in [2, 3]:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_A)
    sizes = np.bincount(labels)
    min_size = sizes.min()
    sil = silhouette_score(X_A, labels)
    ch = calinski_harabasz_score(X_A, labels)
    db = davies_bouldin_score(X_A, labels)
    A_metrics.append({'k': k, 'silhouette': sil, 'CH': ch, 'DB': db,
                      'min_size': min_size, 'sizes': sizes.tolist()})
    print(f'  k={k}: silhouette={sil:.3f}  CH={ch:.1f}  DB={db:.3f}  '
          f'min_size={min_size}  sizes={sizes.tolist()}')

A_metrics_df = pd.DataFrame(A_metrics)

# Bootstrap Jaccard for both k values (since N is smaller, stability is critical)
print()
best_k_A = None
for k in [2, 3]:
    if A_metrics_df[A_metrics_df['k'] == k]['min_size'].values[0] < MIN_CLUSTER_SIZE:
        print(f'  k={k}: skipping stability (min_size < {MIN_CLUSTER_SIZE})')
        continue
    j_mean, j_std = bootstrap_jaccard_stability(X_A, k, n_bootstraps=N_BOOTSTRAPS,
                                                  sample_frac=SAMPLE_FRAC,
                                                  random_state=RANDOM_STATE)
    s_label = 'STABLE' if j_mean >= 0.75 else ('MODERATE' if j_mean >= 0.5 else 'UNSTABLE')
    print(f'  k={k}: Jaccard mean={j_mean:.3f}  std={j_std:.3f}  → {s_label}')
    stability_records.append({'population': 'Profile A', 'k': k,
                               'jaccard_mean': j_mean, 'jaccard_std': j_std, 'stability': s_label})
    if j_mean >= 0.75 and best_k_A is None:
        best_k_A = k
        jacc_mean_A, jacc_std_A = j_mean, j_std
        stability_label_A = s_label

if best_k_A is None:
    print('\n  No stable sub-cluster solution found for Profile A.')
    print('  Profile A is treated as a single homogeneous distributed-network type.')
else:
    print(f'\nSelected k for Profile A: k = {best_k_A} (Jaccard={jacc_mean_A:.3f} — {stability_label_A})')

In [ ]:
# --- Profile A: Final assignment and profiles ---
if best_k_A is not None:
    km_A_final = KMeans(n_clusters=best_k_A, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    profA = profA.copy()
    profA['subcluster_A'] = km_A_final.fit_predict(X_A)
    # Label by H_trans median (ascending)
    cluster_htrans_A = profA.groupby('subcluster_A')['H_trans'].median().sort_values()
    label_map_A = {idx: f'A{i+1}' for i, idx in enumerate(cluster_htrans_A.index)}
    profA['subprofile_A'] = profA['subcluster_A'].map(label_map_A)

    print(f'Profile A sub-cluster sizes:')
    for sp, count in profA['subprofile_A'].value_counts().sort_index().items():
        print(f'  {sp}: N={count} ({count/len(profA):.1%})')

    # Condition χ²
    cond_tab_A = pd.crosstab(profA['subprofile_A'], profA['study_condition'])
    chi2_A, p_cond_A, dof_A, _ = chi2_contingency(cond_tab_A)
    print(f'  Condition χ² = {chi2_A:.3f}, p = {p_cond_A:.4f}')

    # Feature profiles
    print()
    print('Profile A sub-cluster feature profiles (means):')
    for sp in sorted(profA['subprofile_A'].unique()):
        grp = profA[profA['subprofile_A'] == sp]
        means = {c: grp[c].mean() for c in NET_FEATURES + ['sig_hat', 'H_emp', 'x_mean']
                 if c in profA.columns}
        print(f'  {sp} (N={len(grp)}): ' + '  '.join([f'{c}={v:.3f}' for c, v in means.items()]))
else:
    profA = profA.copy()
    profA['subprofile_A'] = 'A (undivided)'
    print('Profile A: No stable sub-structure found. Treated as single distributed-network type.')

In [ ]:
# --- Figure C6: Profile A sub-cluster analysis (condensed) ---
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
fig.suptitle(f'Profile A — Secondary Sub-cluster Analysis\n(N={len(profA)}; k={best_k_A if best_k_A else "none"})',
             fontsize=12)

subprofiles_A = sorted(profA['subprofile_A'].unique())
palette_A = dict(zip(subprofiles_A, ['#70AD47', '#A9D18E', '#548235', '#375623']))

for ax, feat, label in zip(
    axes,
    ['H_trans', 'H_app', 'sig_hat'],
    ['H_trans (transition entropy)', 'H_app (app diversity)', 'σ̂ (noise; overlay only)']
):
    if feat not in profA.columns:
        ax.set_visible(False)
        continue
    data_groups = [profA[profA['subprofile_A'] == sp][feat].dropna() for sp in subprofiles_A]
    positions = range(len(subprofiles_A))
    if any(len(g) > 0 for g in data_groups):
        bp = ax.violinplot([g.values for g in data_groups if len(g) > 0],
                           positions=list(positions)[:len([g for g in data_groups if len(g)>0])],
                           showmedians=True, showextrema=False)
        for pc, sp in zip(bp['bodies'], subprofiles_A):
            pc.set_facecolor(palette_A.get(sp, '#ccc'))
            pc.set_alpha(0.7)
        bp['cmedians'].set_color('black')
    ax.set_xticks(list(positions))
    ax.set_xticklabels([f'{sp}\n(N={len(profA[profA["subprofile_A"]==sp])})'
                        for sp in subprofiles_A], fontsize=9)
    ax.set_title(label, fontsize=9)
    ax.set_ylabel(feat, fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_c6_profileA_subcluster.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_c6_profileA_subcluster.png')

---
## Section 5: Full-sample NeT-only Clustering (Robustness Check)

Run k-means on all N=463 NeT-eligible participants using NeT features only (no behavior features).
Compare the resulting assignments with Phase 1 assignments using ARI.

**Interpretation:** High ARI means the Phase 1 split is reproducible from NeT features alone.
Low ARI means behavior features were contributing more to Phase 1 than it appeared.

In [ ]:
# --- Full-sample NeT-only clustering ---
print(f'=== Full-sample NeT-only Clustering (N={len(full_net)}) ===')
print()

scaler_full = StandardScaler()
X_full = scaler_full.fit_transform(full_net[NET_FEATURES])

full_metrics = []
for k in [2, 3, 4]:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_full)
    sizes = np.bincount(labels)
    sil = silhouette_score(X_full, labels)
    ch = calinski_harabasz_score(X_full, labels)
    db = davies_bouldin_score(X_full, labels)
    full_metrics.append({'k': k, 'silhouette': sil, 'CH': ch, 'DB': db,
                         'min_size': sizes.min(), 'sizes': sizes.tolist()})
    print(f'  k={k}: silhouette={sil:.3f}  CH={ch:.1f}  DB={db:.3f}  sizes={sizes.tolist()}')

full_metrics_df = pd.DataFrame(full_metrics)
best_k_full = int(full_metrics_df.loc[full_metrics_df['silhouette'].idxmax(), 'k'])
print(f'\nBest k (full NeT-only): k = {best_k_full}')

# Bootstrap stability for full-sample
j_full_mean, j_full_std = bootstrap_jaccard_stability(
    X_full, best_k_full, n_bootstraps=N_BOOTSTRAPS,
    sample_frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
s_full_label = 'STABLE' if j_full_mean >= 0.75 else ('MODERATE' if j_full_mean >= 0.5 else 'UNSTABLE')
print(f'  Jaccard: mean={j_full_mean:.3f}  std={j_full_std:.3f}  → {s_full_label}')
stability_records.append({'population': 'Full NeT-eligible', 'k': best_k_full,
                           'jaccard_mean': j_full_mean, 'jaccard_std': j_full_std,
                           'stability': s_full_label})

In [ ]:
# --- ARI comparison: full NeT-only vs. Phase 1 ---
km_full_final = KMeans(n_clusters=best_k_full, init='k-means++', n_init=10, random_state=RANDOM_STATE)
full_net['cluster_net_only'] = km_full_final.fit_predict(X_full)

# Only compare participants who have Phase 1 labels
overlap = full_net[full_net['profile_p1'].notna()].copy()
if len(overlap) > 0:
    # Convert Phase 1 profile to numeric
    overlap['profile_p1_num'] = (overlap['profile_p1'] == 'Profile B').astype(int)
    ari_full_vs_p1 = adjusted_rand_score(overlap['profile_p1_num'],
                                          overlap['cluster_net_only'])
    print(f'\nARI (full NeT-only k={best_k_full} vs. Phase 1 k=2): {ari_full_vs_p1:.3f}')
    if ari_full_vs_p1 >= 0.7:
        print('  ✓ HIGH ARI: Phase 1 network structure split is largely reproducible')
        print('    from NeT features alone — behavior features amplified but did not create the split.')
    elif ari_full_vs_p1 >= 0.4:
        print('  ~ MODERATE ARI: Phase 1 structure is partially reproducible from NeT features.')
        print('    Behavior features contributed meaningfully to Phase 1 cluster boundaries.')
    else:
        print('  ⚠ LOW ARI: Phase 1 split is substantially different from NeT-only solution.')
        print('    Behavior features drove the Phase 1 cluster boundaries significantly.')
    
    # Show how Phase 1 profiles distribute across NeT-only clusters
    print()
    print('Phase 1 profile vs. NeT-only cluster cross-tabulation:')
    ct = pd.crosstab(overlap['profile_p1'], overlap['cluster_net_only'])
    print(ct)
else:
    print('No overlap between NeT-eligible and Phase 1 populations found.')
    ari_full_vs_p1 = np.nan

In [ ]:
# --- Figure C7: Full-sample NeT-only clustering ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f'Full NeT-eligible Sample — NeT-only Clustering (N={len(full_net)})\n'
             f'Robustness check for Phase 1 split', fontsize=12)

# Left: silhouette by k for full sample
ax = axes[0]
ks_f = full_metrics_df['k']
colors_f = ['#4472C4' if k == best_k_full else '#A9C4E8' for k in ks_f]
ax.bar(ks_f, full_metrics_df['silhouette'], color=colors_f)
ax.set_xlabel('k'); ax.set_ylabel('Silhouette Score')
ax.set_title(f'Full NeT-only k-selection\n(best k={best_k_full})')
ax.set_xticks(ks_f)
for k, v in zip(ks_f, full_metrics_df['silhouette']):
    ax.text(k, v + 0.002, f'{v:.3f}', ha='center', fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Right: PCA scatter of full sample, colored by NeT-only cluster
ax = axes[1]
pca_full = PCA(n_components=2)
X_full_pca = pca_full.fit_transform(X_full)
palette_full = ['#4472C4', '#ED7D31', '#70AD47', '#FFC000']
cluster_labels_full = full_net['cluster_net_only'].values
for c in sorted(np.unique(cluster_labels_full)):
    mask_c = cluster_labels_full == c
    ax.scatter(X_full_pca[mask_c, 0], X_full_pca[mask_c, 1],
               c=palette_full[c], alpha=0.5, s=15, label=f'NeT cluster {c}')
# Overlay Phase 1 labels where available
if len(overlap) > 0:
    overlap_idx = full_net['profile_p1'].notna().values
    phase1_B = full_net.loc[full_net['profile_p1'] == 'Profile B'].index
    p1_idx = np.array([i for i, pid in enumerate(full_net['participant_id_canonical'])
                       if full_net.loc[full_net['participant_id_canonical']==pid, 'profile_p1'].values[0] == 'Profile B'
                       if len(full_net.loc[full_net['participant_id_canonical']==pid, 'profile_p1'].values) > 0])

ax.set_xlabel(f'PC1 ({pca_full.explained_variance_ratio_[0]*100:.0f}% var)')
ax.set_ylabel(f'PC2 ({pca_full.explained_variance_ratio_[1]*100:.0f}% var)')
ax.set_title(f'PCA scatter (NeT-only clusters)\nARI vs Phase 1 = {ari_full_vs_p1:.3f}')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_c7_fullsample_net_only.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_c7_fullsample_net_only.png')

---
## Section 6: H_pi Supplementary Sub-analysis

H_pi (stationary distribution entropy) is available for participants with ≥50 transitions
and an ergodic transition matrix. Within Profile B, 278/289 (96%) are stationary-eligible.

This supplementary analysis tests whether adding H_pi changes the Profile B sub-cluster structure.

In [ ]:
# --- H_pi supplementary sub-analysis for Profile B ---
print('=== H_pi Supplementary Sub-analysis (Profile B) ===')

profB_hpi = profB.dropna(subset=['H_pi']).copy()
print(f'Profile B with H_pi available: N = {len(profB_hpi)} '
      f'({len(profB_hpi)/len(profB):.1%} of Profile B)')

FEATURES_HAPI = NET_FEATURES + ['H_pi']
scaler_Bhpi = StandardScaler()
X_Bhpi = scaler_Bhpi.fit_transform(profB_hpi[FEATURES_HAPI])

# Run k=best_k_B on H_pi-inclusive feature set
km_hpi = KMeans(n_clusters=best_k_B, init='k-means++', n_init=10, random_state=RANDOM_STATE)
labels_hpi = km_hpi.fit_predict(X_Bhpi)

# Compare with H_pi-exclusive solution (on same subset)
X_Bhpi_excl = scaler_B.transform(profB_hpi[NET_FEATURES])
km_excl = KMeans(n_clusters=best_k_B, init='k-means++', n_init=10, random_state=RANDOM_STATE)
labels_excl = km_excl.fit_predict(X_Bhpi_excl)

ari_hpi = adjusted_rand_score(labels_excl, labels_hpi)
print(f'ARI (H_pi-inclusive vs. H_pi-exclusive, k={best_k_B}): {ari_hpi:.3f}')
if ari_hpi >= 0.85:
    print('  ✓ H_pi does NOT change the Profile B sub-cluster structure.')
    print('    The 4-feature solution is stable to the addition of stationary entropy.')
elif ari_hpi >= 0.5:
    print('  ~ H_pi partially modifies the sub-cluster structure (moderate ARI).')
    print('    Some participants are reassigned when stationary entropy is included.')
else:
    print('  ⚠ H_pi substantially changes the sub-cluster structure (low ARI).')
    print('    Stationary distribution entropy reveals different structure than transition entropy.')

# H_pi by sub-cluster (from H_pi-exclusive solution, descriptive)
profB_hpi_temp = profB_hpi.copy()
profB_hpi_temp['subcl_for_hpi'] = labels_excl
cluster_htrans_hpi = profB_hpi_temp.groupby('subcl_for_hpi')['H_trans'].median().sort_values()
label_map_hpi = {idx: f'B{i+1}' for i, idx in enumerate(cluster_htrans_hpi.index)}
profB_hpi_temp['subprofile_hpi'] = profB_hpi_temp['subcl_for_hpi'].map(label_map_hpi)
print('\nH_pi medians by sub-cluster (descriptive):')
for sp, grp in profB_hpi_temp.groupby('subprofile_hpi'):
    print(f'  {sp}: H_pi median={grp["H_pi"].median():.4f}  (N={len(grp)})')

---
## Section 7: Save All Outputs

In [ ]:
# --- Build the unified assignment file ---
# Columns: participant_id_canonical, study_condition, profile_p1,
#          subprofile_B (for Profile B members), subprofile_A (for Profile A members),
#          cluster_net_only (full NeT-only solution)

# Merge Profile B sub-labels back to master
assignments = df[['participant_id_canonical', 'study_condition', 'profile_p1']].copy()

# Add Profile B sub-profile
b_label_cols = profB[['participant_id_canonical', 'subprofile_B']]
assignments = assignments.merge(b_label_cols, on='participant_id_canonical', how='left')

# Add Profile A sub-profile
a_label_cols = profA[['participant_id_canonical', 'subprofile_A']]
assignments = assignments.merge(a_label_cols, on='participant_id_canonical', how='left')

# Add full NeT-only cluster
full_net_labels = full_net[['participant_id_canonical', 'cluster_net_only']]
assignments = assignments.merge(full_net_labels, on='participant_id_canonical', how='left')

print(f'Assignment file shape: {assignments.shape}')
print('Sub-profile_B counts (within Profile B):')
print(assignments['subprofile_B'].value_counts(dropna=False).to_dict())
print('Sub-profile_A counts (within Profile A):')
print(assignments['subprofile_A'].value_counts(dropna=False).to_dict())

# Save
outfile_assignments = PHENOTYPING_DIR / 'ds5_net_subcluster_assignments.csv'
assignments.to_csv(outfile_assignments, index=False)
print(f'\nSaved: {outfile_assignments}')

In [ ]:
# --- Build the unified profiles file ---
# One row per sub-cluster, with feature means and medians
profile_summary_rows = []

# Profile B sub-clusters
for sp in sorted(profB['subprofile_B'].unique()):
    grp = profB[profB['subprofile_B'] == sp]
    row = {'source_population': 'Profile B', 'subprofile': sp, 'N': len(grp)}
    for col in ALL_PROFILE_COLS:
        if col in grp.columns:
            row[f'{col}_mean'] = grp[col].mean()
            row[f'{col}_median'] = grp[col].median()
            row[f'{col}_std'] = grp[col].std()
    profile_summary_rows.append(row)

# Profile A sub-clusters (if found)
a_col = 'subprofile_A'
if a_col in profA.columns:
    for sp in sorted(profA[a_col].unique()):
        grp = profA[profA[a_col] == sp]
        row = {'source_population': 'Profile A', 'subprofile': sp, 'N': len(grp)}
        for col in ALL_PROFILE_COLS:
            if col in grp.columns:
                row[f'{col}_mean'] = grp[col].mean()
                row[f'{col}_median'] = grp[col].median()
                row[f'{col}_std'] = grp[col].std()
        profile_summary_rows.append(row)

profiles_out_df = pd.DataFrame(profile_summary_rows)

outfile_profiles = PHENOTYPING_DIR / 'ds5_net_subcluster_profiles.csv'
profiles_out_df.to_csv(outfile_profiles, index=False)
print(f'Saved: {outfile_profiles}')
print(f'Profiles file shape: {profiles_out_df.shape}')

In [ ]:
# --- Save stability records ---
stability_df = pd.DataFrame(stability_records)
outfile_stability = PHENOTYPING_DIR / 'ds5_net_subcluster_stability.csv'
stability_df.to_csv(outfile_stability, index=False)
print(f'Saved: {outfile_stability}')
print(stability_df.to_string(index=False))

In [ ]:
# --- Build and save model_run_summary ---
import datetime

# Collect key numbers
b1_n = len(profB[profB['subprofile_B'] == 'B1']) if 'subprofile_B' in profB.columns else 0
b2_n = len(profB[profB['subprofile_B'] == 'B2']) if 'subprofile_B' in profB.columns else 0

run_summary_lines = [
    '# DS5 Step 2 Path C — Network Sub-Phenotyping: Model Run Summary',
    '',
    f'**Date:** {datetime.date.today()}',
    f'**Notebook:** 101_ds5_network_subphenotype.ipynb',
    f'**Stage:** Stage 4, Step 2 — Path C',
    '',
    '---',
    '',
    '## Run parameters',
    f'- NeT clustering features: {NET_FEATURES}',
    f'- RANDOM_STATE: {RANDOM_STATE}',
    f'- Bootstrap: {N_BOOTSTRAPS} iterations, {int(SAMPLE_FRAC*100)}% sample fraction',
    f'- Min cluster size threshold: {MIN_CLUSTER_SIZE}',
    '',
    '## Analysis populations',
    f'- Profile B (primary): N = {len(profB)}',
    f'- Profile A (secondary): N = {len(profA)}',
    f'- Full NeT-eligible (robustness): N = {len(full_net)}',
    '',
    '## Decision gate',
    f'- Gate triggered: {decision_gate_triggered}',
    f'- Profile B H_trans=0: {n_zero} participants ({pct_htrans_zero:.1%}) — zero-inflated',
    f'- Profile B H_trans>0: {n_nonzero} participants ({1-pct_htrans_zero:.1%})',
    '',
    '## Profile B sub-clustering',
    f'- Selected k: {best_k_B}',
    f'- Silhouette: {B_metrics_df[B_metrics_df["k"]==best_k_B]["silhouette"].values[0]:.3f}',
    f'- Bootstrap Jaccard: {jacc_mean_B:.3f} ± {jacc_std_B:.3f} ({stability_label_B})',
    f'- Condition χ²: {chi2:.3f}, p={p_cond:.4f}',
    f'- Psychosocial significant (after Bonferroni): {n_sig_B} of {len(psych_results_B)}',
]

# Add sub-profile means
for sp in sorted(profB['subprofile_B'].unique()):
    grp = profB[profB['subprofile_B'] == sp]
    run_summary_lines.append(f'')
    run_summary_lines.append(f'### {sp} (N={len(grp)})')
    for col in NET_FEATURES:
        if col in grp.columns:
            run_summary_lines.append(f'- {col}: mean={grp[col].mean():.4f}, median={grp[col].median():.4f}')
    for col in ['sig_hat', 'H_emp', 'x_mean']:
        if col in grp.columns:
            run_summary_lines.append(f'- {col} (overlay): mean={grp[col].mean():.4f}, median={grp[col].median():.4f}')

run_summary_lines += [
    '',
    '## Profile A sub-clustering',
    f'- Selected k: {best_k_A if best_k_A else "none (no stable solution)"}',
]
if best_k_A:
    run_summary_lines.append(f'- Bootstrap Jaccard: {jacc_mean_A:.3f} ± {jacc_std_A:.3f} ({stability_label_A})')

run_summary_lines += [
    '',
    '## Full-sample NeT-only robustness',
    f'- Best k: {best_k_full}',
    f'- Bootstrap Jaccard: {j_full_mean:.3f} ± {j_full_std:.3f} ({s_full_label})',
    f'- ARI vs Phase 1: {ari_full_vs_p1:.3f}',
    '',
    '## H_pi supplementary',
    f'- ARI (H_pi-inclusive vs. H_pi-exclusive, k={best_k_B}): {ari_hpi:.3f}',
    '',
    '## Output files',
    '- ds5_net_subcluster_assignments.csv',
    '- ds5_net_subcluster_profiles.csv',
    '- ds5_net_subcluster_stability.csv',
    '- figures/fig_c1_profileB_net_distributions.png',
    '- figures/fig_c2_profileB_pca.png',
    '- figures/fig_c3_profileB_subcluster_selection.png',
    '- figures/fig_c4_profileB_subcluster_profiles.png',
    '- figures/fig_c5_profileB_overlay.png',
    '- figures/fig_c6_profileA_subcluster.png',
    '- figures/fig_c7_fullsample_net_only.png',
    '',
    '## Key limitations',
    '1. DS5 networks are built from OneSec-triggered events only — not total app usage',
    '2. Temporal succession (app_i → app_j) is not causal coupling',
    '3. First-order Markov property is not verified for DS5',
    '4. H_pi is an approximation given short windows (7–45 days)',
    '5. Zero-inflation in H_trans/H_app creates a discrete (not continuous) sub-structure',
    '6. All findings are exploratory and DS5-specific; cross-dataset replication required',
]

run_summary_text = '\n'.join(run_summary_lines)

outfile_run_summary = PHENOTYPING_DIR / 'model_run_summary_path_c.md'
with open(outfile_run_summary, 'w') as f:
    f.write(run_summary_text)
print(f'Saved: {outfile_run_summary}')
print()
print(run_summary_text)

---
## Summary and Interpretation

### What was found

**Profile B sub-structure:** The primary finding is that H_trans and H_app are zero-inflated
within Profile B (70.6% at exactly 0). This creates a discrete behavioral distinction between:
- **B1** — Pure single-app users: H_trans=0, H_app=0; every OneSec event targets the same app;
  network is a single self-loop with no observable inter-app transitions
- **B2** — Multi-app hub-locked users: H_trans>0, H_app>0 but HHI still near ceiling;
  primarily one dominant app with occasional transitions to a small secondary set

**Important caveat:** This sub-structure is driven by whether the participant ever opened
a second app through OneSec. Because DS5 only logs OneSec-triggered events (not total phone use),
a B1 participant may use other apps extensively — they simply never triggered OneSec on them.
B1 and B2 describe network structure in the OneSec behavioral record, not in total app usage.

**Profile A sub-structure:** Profile A has continuous H_trans/H_app distribution.
Whether sub-structure was found depends on stability results above.

**Full-sample NeT-only:** The ARI vs. Phase 1 indicates the degree to which Phase 1
behavioral features were driving or merely amplifying the network-structure split.

### Step 2 implications

- Path C is complete.
- Path A (behavior-only phenotyping without NeT) is the next recommended analysis.
- The B1/B2 sub-types feed directly into Path F (hierarchical phenotype taxonomy).
- The condition-independence of B1/B2 (if confirmed) supports treating network sub-types
  as behavioral trait-level distinctions rather than experimental artifacts.

### What this is NOT
- This is not Step 3. No intervention is prescribed for any sub-cluster.
- The B1/B2 distinction does not imply which friction type works best — that is Step 3.
- These findings are exploratory and DS5-specific. Cross-dataset replication (DS4, DS6)
  is required before claiming generalizability.